# NB03d — Multimodal Alignment and Governance Review

This notebook is the **alignment layer** for Phase 4. It does not run any new model.
Instead, it asks a narrower and more useful question:

> when Qwen-based menu understanding and frozen V-JEPA structure signals are placed side by
> side, what do they agree on, where do they disagree, and which images should be kept,
> reviewed, or dropped for future dataset building?

This is the notebook that turns `NB03b` and `NB03c` from separate outputs into one
governance-oriented evidence package.

Main outputs:

- `data/processed/menu_image_subset_v1_multimodal_alignment.csv`
- `data/processed/menu_image_subset_v1_governance_table.csv`
- `outputs/figures/nb03d/fig28_multimodal_governance_map.png`
- `outputs/figures/nb03d/fig29_vjepa_nearest_neighbor_panel.png`

This is still a **small-N evidence notebook**. The goal is not to prove robust multimodal
classification. The goal is to document which pages look analytically promising, fragile,
or exclusion-worthy inside the current sandbox subset.

## 1. Environment setup

This notebook consumes already-saved outputs only:

- the curated subset from `NB03a`
- Qwen understanding outputs from `NB03b`
- V-JEPA embedding summaries from `NB03c`

No model is loaded here. This keeps the alignment pass cheap, auditable, and easy to
rerun while the analytical story is still evolving.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

PROCESSED_DIR = ROOT / "data" / "processed"
SUBSET_PATH = PROCESSED_DIR / "menu_image_subset_v1.csv"
QWEN_PATH = PROCESSED_DIR / "menu_image_subset_v1_qwen_understanding.csv"
EMBED_INDEX_CSV = PROCESSED_DIR / "menu_image_subset_v1_vjepa_embedding_index.csv"
ALIGNMENT_CSV = PROCESSED_DIR / "menu_image_subset_v1_multimodal_alignment.csv"
GOVERNANCE_CSV = PROCESSED_DIR / "menu_image_subset_v1_governance_table.csv"
FIG_DIR = ROOT / "outputs" / "figures" / "nb03d"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for path in [SUBSET_PATH, QWEN_PATH, EMBED_INDEX_CSV]:
    assert path.exists(), f"Required input not found: {path}"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print(f"Project root    : {ROOT}")
print(f"Subset path     : {SUBSET_PATH}")
print(f"Qwen path       : {QWEN_PATH}")
print(f"Embedding index : {EMBED_INDEX_CSV}")
print(f"Alignment CSV   : {ALIGNMENT_CSV}")
print(f"Governance CSV  : {GOVERNANCE_CSV}")
print(f"Figure dir      : {FIG_DIR}")

## 2. Load and merge the Phase 4 evidence layers

The merged frame keeps three kinds of information together:

- source/curation metadata from `NB03a`
- semantic understanding signals from `NB03b`
- structural/neighbourhood signals from `NB03c`

The point is not to find a perfect unified score. It is to see where these layers point in
the same direction and where they diverge.

In [ ]:
subset = pd.read_csv(SUBSET_PATH).copy()
qwen = pd.read_csv(QWEN_PATH).copy()
embed = pd.read_csv(EMBED_INDEX_CSV).copy()


def normalize_qwen_value(value, fallback: str = "missing") -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return fallback
    value = str(value).strip().lower()
    if not value:
        return fallback
    return {
        "moderate": "medium",
        "restaurant menu": "menu_page",
    }.get(value, value)


qwen["run_status_plot"] = qwen["run_status"].map(lambda x: normalize_qwen_value(x, fallback="missing"))
qwen["page_type_plot"] = qwen["page_type"].map(normalize_qwen_value)
qwen["extraction_utility_plot"] = qwen["extraction_utility"].map(normalize_qwen_value)

alignment_df = (
    subset.merge(
        qwen[
            [
                "path",
                "model_name",
                "ocr_like_text",
                "dense_caption",
                "layout_note",
                "style_tags",
                "page_type",
                "extraction_utility",
                "quality_confidence_note",
                "run_status",
                "elapsed_s",
                "error",
                "run_status_plot",
                "page_type_plot",
                "extraction_utility_plot",
            ]
        ],
        on="path",
        how="left",
        suffixes=("", "_qwen"),
    )
    .merge(
        embed[
            [
                "path",
                "input_frames",
                "embedding_dim",
                "embedding_norm",
                "mean_cosine_to_others",
                "nearest_neighbor_filename",
                "nearest_neighbor_score",
            ]
        ],
        on="path",
        how="left",
    )
)

print(f"Aligned rows: {len(alignment_df)}")
print()
display(
    alignment_df[
        [
            "source",
            "subset_role",
            "filename",
            "run_status_plot",
            "page_type_plot",
            "extraction_utility_plot",
            "mean_cosine_to_others",
            "nearest_neighbor_filename",
            "nearest_neighbor_score",
        ]
    ]
)


## 3. Derive alignment buckets and governance decisions

The governance logic is intentionally transparent rather than clever.

It uses:

- **Qwen usefulness** as a semantic utility signal
- **mean cosine similarity** as a visual centrality / outlier signal
- **nearest neighbour** as a local coherence check

The resulting `keep / review / drop` recommendation is a sandbox governance aid, not a
production classifier.

In [ ]:
utility_rank = {"missing": -1, "low": 0, "medium": 1, "high": 2}
page_type_rank = {"ambiguous": 0, "menu_adjacent": 1, "menu_page": 2, "missing": -1}

alignment_df["utility_score"] = alignment_df["extraction_utility_plot"].map(utility_rank).fillna(-1).astype(int)
alignment_df["page_type_score"] = alignment_df["page_type_plot"].map(page_type_rank).fillna(-1).astype(int)
alignment_df["qwen_ok"] = alignment_df["run_status_plot"].eq("ok")
alignment_df["qwen_useful"] = alignment_df["extraction_utility_plot"].isin(["high", "medium"])
alignment_df["likely_menu_page"] = alignment_df["page_type_plot"].eq("menu_page")

mean_cos_q25 = float(alignment_df["mean_cosine_to_others"].quantile(0.25))
mean_cos_q75 = float(alignment_df["mean_cosine_to_others"].quantile(0.75))
mean_cos_median = float(alignment_df["mean_cosine_to_others"].median())


def visual_bucket(value: float) -> str:
    if value <= mean_cos_q25:
        return "outlier_side"
    if value >= mean_cos_q75:
        return "central_side"
    return "mid_band"


alignment_df["visual_bucket"] = alignment_df["mean_cosine_to_others"].map(visual_bucket)

neighbor_lookup = (
    alignment_df[
        [
            "filename",
            "source",
            "run_status_plot",
            "page_type_plot",
            "extraction_utility_plot",
            "mean_cosine_to_others",
        ]
    ]
    .rename(columns={
        "filename": "nearest_neighbor_filename",
        "source": "neighbor_source",
        "run_status_plot": "neighbor_run_status",
        "page_type_plot": "neighbor_page_type",
        "extraction_utility_plot": "neighbor_extraction_utility",
        "mean_cosine_to_others": "neighbor_mean_cosine",
    })
)

alignment_df = alignment_df.merge(neighbor_lookup, on="nearest_neighbor_filename", how="left")
alignment_df["neighbor_utility_match"] = (
    alignment_df["extraction_utility_plot"] == alignment_df["neighbor_extraction_utility"]
)
alignment_df["neighbor_page_type_match"] = (
    alignment_df["page_type_plot"] == alignment_df["neighbor_page_type"]
)


def classify_alignment(row: pd.Series) -> str:
    if row["qwen_ok"] and row["qwen_useful"] and row["likely_menu_page"] and row["visual_bucket"] != "outlier_side":
        return "aligned_promising"
    if (not row["qwen_ok"] or row["extraction_utility_plot"] in ["missing", "low"]) and row["visual_bucket"] == "central_side":
        return "semantic_gap_review"
    if row["qwen_useful"] and row["visual_bucket"] == "outlier_side":
        return "visual_novelty_review"
    if row["page_type_plot"] == "ambiguous" and row["extraction_utility_plot"] in ["low", "missing"] and row["visual_bucket"] == "outlier_side":
        return "aligned_exclusion"
    if not row["qwen_ok"] and row["visual_bucket"] == "outlier_side":
        return "unstable_exclusion"
    return "mixed_review"


def classify_governance(row: pd.Series) -> str:
    if row["alignment_bucket"] == "aligned_promising":
        return "keep"
    if row["alignment_bucket"] in ["aligned_exclusion", "unstable_exclusion"]:
        return "drop"
    return "review"


def governance_reason(row: pd.Series) -> str:
    if row["alignment_bucket"] == "aligned_promising":
        return "Qwen sees a usable menu page and JEPA places it away from the outlier edge."
    if row["alignment_bucket"] == "semantic_gap_review":
        return "Visually central or coherent page, but Qwen utility is weak or the run failed."
    if row["alignment_bucket"] == "visual_novelty_review":
        return "Qwen finds menu value, but JEPA places the page near the visual edge of the subset."
    if row["alignment_bucket"] == "aligned_exclusion":
        return "Ambiguous or non-menu page with weak utility and outlier-side visual behaviour."
    if row["alignment_bucket"] == "unstable_exclusion":
        return "Qwen failed and JEPA also places the page near the outlier edge."
    return "Signals are mixed; retain for manual review rather than hard keep/drop."


alignment_df["alignment_bucket"] = alignment_df.apply(classify_alignment, axis=1)
alignment_df["governance_decision"] = alignment_df.apply(classify_governance, axis=1)
alignment_df["governance_reason"] = alignment_df.apply(governance_reason, axis=1)

alignment_df = alignment_df.sort_values(
    ["governance_decision", "alignment_bucket", "mean_cosine_to_others"],
    ascending=[True, True, True],
).reset_index(drop=True)

alignment_df.to_csv(ALIGNMENT_CSV, index=False)

governance_df = alignment_df[
    [
        "source",
        "subset_role",
        "filename",
        "run_status_plot",
        "page_type_plot",
        "extraction_utility_plot",
        "mean_cosine_to_others",
        "visual_bucket",
        "nearest_neighbor_filename",
        "nearest_neighbor_score",
        "neighbor_extraction_utility",
        "neighbor_utility_match",
        "alignment_bucket",
        "governance_decision",
        "governance_reason",
        "dense_caption",
        "layout_note",
        "quality_confidence_note",
        "manual_note",
    ]
].copy()

governance_df.to_csv(GOVERNANCE_CSV, index=False)

print({
    "mean_cos_q25": round(mean_cos_q25, 4),
    "mean_cos_median": round(mean_cos_median, 4),
    "mean_cos_q75": round(mean_cos_q75, 4),
})
print()
print("Governance decisions:")
print(governance_df["governance_decision"].value_counts().to_string())
print()
print("Alignment buckets:")
print(governance_df["alignment_bucket"].value_counts().to_string())
print()

display(governance_df)
print(f"Saved alignment CSV  : {ALIGNMENT_CSV}")
print(f"Saved governance CSV : {GOVERNANCE_CSV}")

## 4. Review the most important mismatch cases

These are analytically the most useful rows for the horizon scan:

- visually central pages where Qwen still struggled
- visually unusual pages that nevertheless produced useful menu understanding
- coherent nearest-neighbour pairs with different semantic outcomes

Those cases are evidence that **visual structure and semantic usefulness are related but not
identical** in this sandbox.

In [ ]:
review_cases = governance_df.loc[
    governance_df["governance_decision"].eq("review")
].copy()

review_cases["review_priority"] = np.select(
    [
        review_cases["alignment_bucket"].eq("semantic_gap_review"),
        review_cases["alignment_bucket"].eq("visual_novelty_review"),
    ],
    [0, 1],
    default=2,
)

review_cases = review_cases.sort_values(
    ["review_priority", "mean_cosine_to_others", "nearest_neighbor_score"],
    ascending=[True, True, False],
).reset_index(drop=True)

display(
    review_cases[
        [
            "source",
            "filename",
            "alignment_bucket",
            "run_status_plot",
            "page_type_plot",
            "extraction_utility_plot",
            "mean_cosine_to_others",
            "nearest_neighbor_filename",
            "nearest_neighbor_score",
            "neighbor_extraction_utility",
            "governance_reason",
        ]
    ]
)


## 5. Figure 28 — Multimodal governance map

This scatter does not claim a universal metric. It is simply a compact way to show how the
current subset sits between:

- **semantic usefulness** on the y-axis
- **visual centrality** on the x-axis

The coloured recommendation labels make the governance logic auditable rather than hidden
inside prose.

In [ ]:
fig, ax = plt.subplots(figsize=(10.6, 6.2))

sns.scatterplot(
    data=alignment_df,
    x="mean_cosine_to_others",
    y="utility_score",
    hue="governance_decision",
    style="run_status_plot",
    s=130,
    ax=ax,
)

for _, row in alignment_df.iterrows():
    ax.text(
        row["mean_cosine_to_others"] + 0.002,
        row["utility_score"] + 0.03,
        row["filename"][:18],
        fontsize=7,
    )

ax.axvline(mean_cos_median, color="gray", linestyle="--", linewidth=1)
ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
ax.set_title("Multimodal governance map: JEPA centrality vs Qwen utility")
ax.set_xlabel("mean cosine similarity to the rest of the subset")
ax.set_ylabel("Qwen extraction utility score")
ax.set_yticks([-1, 0, 1, 2])
ax.set_yticklabels(["missing", "low", "medium", "high"])

if ax.get_legend() is not None:
    try:
        sns.move_legend(
            ax,
            "upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
            fontsize=8,
            title_fontsize=9,
        )
    except Exception:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles,
            labels,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            borderaxespad=0.0,
            frameon=True,
            fontsize=8,
            title_fontsize=9,
        )

fig.tight_layout(rect=(0, 0, 0.82, 1))

fig28 = FIG_DIR / "fig28_multimodal_governance_map.png"
fig.savefig(fig28)
plt.show()

print(f"Saved figure: {fig28}")

## 6. Figure 29 — V-JEPA nearest-neighbour review panel

The panel below focuses on the strongest local neighbour pairs found in the frozen JEPA
space. It is not intended as a statistical summary. It is a visual audit aid:

- do the paired pages look plausibly related?
- do their Qwen outcomes agree?
- where does visual similarity still fail to guarantee semantic usefulness?

In [ ]:
filename_to_path = dict(zip(alignment_df["filename"], alignment_df["path"]))
filename_to_decision = dict(zip(alignment_df["filename"], alignment_df["governance_decision"]))
filename_to_utility = dict(zip(alignment_df["filename"], alignment_df["extraction_utility_plot"]))
filename_to_status = dict(zip(alignment_df["filename"], alignment_df["run_status_plot"]))
filename_to_source = dict(zip(alignment_df["filename"], alignment_df["source"]))

pair_df = alignment_df[["filename", "nearest_neighbor_filename", "nearest_neighbor_score"]].copy()
pair_df["pair_key"] = pair_df.apply(
    lambda row: " || ".join(sorted([row["filename"], row["nearest_neighbor_filename"]])),
    axis=1,
)
pair_df = pair_df.sort_values("nearest_neighbor_score", ascending=False)
pair_df = pair_df.drop_duplicates("pair_key").head(4).reset_index(drop=True)

fig, axes = plt.subplots(len(pair_df), 2, figsize=(10.5, 3.2 * max(len(pair_df), 1)))
if len(pair_df) == 1:
    axes = np.array([axes])

for row_idx, pair in pair_df.iterrows():
    for col_idx, filename in enumerate([pair["filename"], pair["nearest_neighbor_filename"]]):
        ax = axes[row_idx, col_idx]
        image_path = Path(filename_to_path[filename])
        with Image.open(image_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            ax.imshow(im)
        ax.set_axis_off()
        ax.set_title(
            "\n".join(
                [
                    filename[:44],
                    f"{filename_to_source[filename]} | {filename_to_status[filename]} | {filename_to_utility[filename]}",
                    f"decision: {filename_to_decision[filename]}",
                ]
            ),
            fontsize=8,
        )
    axes[row_idx, 0].set_ylabel(
        f"pair score\n{pair['nearest_neighbor_score']:.3f}",
        rotation=0,
        labelpad=35,
        fontsize=8,
        va="center",
    )

fig.suptitle("Nearest-neighbour review panel from the frozen V-JEPA space", fontsize=13, y=1.01)
fig.tight_layout()

fig29 = FIG_DIR / "fig29_vjepa_nearest_neighbor_panel.png"
fig.savefig(fig29)
plt.show()

print(f"Saved figure: {fig29}")

## 7. Short audit note for the sandbox narrative

This final cell turns the notebook outputs into direct interpretive prompts rather than a
long free-form conclusion. The writeup should stay close to these observations.

In [ ]:
keep_n = int((governance_df["governance_decision"] == "keep").sum())
review_n = int((governance_df["governance_decision"] == "review").sum())
drop_n = int((governance_df["governance_decision"] == "drop").sum())

print(f"Keep  : {keep_n}")
print(f"Review: {review_n}")
print(f"Drop  : {drop_n}")
print()

print("Interpretation prompts:")
print("- Which retained pages are both visually central and semantically useful?")
print("- Which visually coherent pages still failed Qwen, and why?")
print("- Which outliers are valuable edge cases versus low-value exclusions?")
print()

print("Writeup guardrail:")
print("Treat this notebook as governance evidence for a tiny curated corpus, not as proof of robust multimodal classification or autonomous design capability.")